# YOLOv8s Training Notebook for Google Colab and Kaggle

This notebook is designed to run on both Google Colab and Kaggle with T4 GPUs.

Pipeline:
- prepare a writable project directory on Colab or Kaggle
- install Ultralytics YOLO
- convert `public/annotations/train.json` and `public/annotations/val.json` to YOLO labels
- train `yolov8s.pt`
- validate the trained model
- export predictions back to the project JSON format
- export error images and `img_error/img_error.json`
- copy `yolov8s.pt`, `best.pt`, and `last.pt` into `models/`

In [ ]:
import os
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_URL = "https://github.com/sinh2206/Object_Detection.git"
REPO_NAME = "Object_Detection"


def looks_like_repo(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "Object_Detection.ipynb").exists()
        and (path / "public" / "classes.json").exists()
        and (path / "public" / "annotations" / "train.json").exists()
        and (path / "public" / "annotations" / "val.json").exists()
    )


def find_existing_repo() -> Path | None:
    direct_candidates = [
        Path.cwd(),
        Path.cwd() / REPO_NAME,
        Path("/content") / REPO_NAME,
        Path("/kaggle/working") / REPO_NAME,
    ]

    for candidate in direct_candidates:
        if looks_like_repo(candidate):
            return candidate.resolve()

    kaggle_input_root = Path("/kaggle/input")
    if kaggle_input_root.exists():
        for classes_path in kaggle_input_root.rglob("classes.json"):
            candidate = classes_path.parent.parent
            if looks_like_repo(candidate):
                return candidate.resolve()

    return None


IS_COLAB = Path("/content").exists() and "COLAB_GPU" in os.environ
IS_KAGGLE = Path("/kaggle/working").exists() and "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_COLAB:
    TARGET_REPO_DIR = Path("/content") / REPO_NAME
elif IS_KAGGLE:
    TARGET_REPO_DIR = Path("/kaggle/working") / REPO_NAME
else:
    TARGET_REPO_DIR = Path.cwd()

existing_repo = find_existing_repo()

if looks_like_repo(TARGET_REPO_DIR):
    REPO_DIR = TARGET_REPO_DIR.resolve()
elif existing_repo is not None and existing_repo != TARGET_REPO_DIR:
    if TARGET_REPO_DIR.exists():
        shutil.rmtree(TARGET_REPO_DIR)
    shutil.copytree(existing_repo, TARGET_REPO_DIR)
    REPO_DIR = TARGET_REPO_DIR.resolve()
elif existing_repo is not None:
    REPO_DIR = existing_repo.resolve()
elif IS_COLAB or IS_KAGGLE:
    !git clone {REPO_URL} {TARGET_REPO_DIR}
    REPO_DIR = TARGET_REPO_DIR.resolve()
else:
    raise FileNotFoundError("Could not locate the Object_Detection project directory.")

os.chdir(REPO_DIR)
print(f"Runtime: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Working directory: {REPO_DIR}")
if (REPO_DIR / '.git').exists():
    !git -C {REPO_DIR} log -1 --oneline
else:
    print("Git metadata is not available in this runtime copy.")

In [ ]:
%pip install -q ultralytics pyyaml matplotlib pillow tqdm

import json
import random
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

In [ ]:
PUBLIC_DIR = REPO_DIR / "public"
MODELS_DIR = REPO_DIR / "models"
ANNOTATIONS_DIR = PUBLIC_DIR / "annotations"
ERROR_DIR = REPO_DIR / "img_error"
ERROR_JSON = ERROR_DIR / "img_error.json"

TRAIN_JSON = ANNOTATIONS_DIR / "train.json"
VAL_JSON = ANNOTATIONS_DIR / "val.json"
CLASSES_JSON = PUBLIC_DIR / "classes.json"

TRAIN_IMAGE_DIR = PUBLIC_DIR / "train" / "images"
VAL_IMAGE_DIR = PUBLIC_DIR / "val" / "images"
TRAIN_LABEL_DIR = PUBLIC_DIR / "train" / "labels"
VAL_LABEL_DIR = PUBLIC_DIR / "val" / "labels"
DATA_YAML = PUBLIC_DIR / "dataset_yolov8.yaml"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
ERROR_DIR.mkdir(parents=True, exist_ok=True)

NUM_GPUS = torch.cuda.device_count()
TRAIN_DEVICE = list(range(NUM_GPUS)) if NUM_GPUS > 1 else (0 if NUM_GPUS == 1 else "cpu")
INFER_DEVICE = 0 if NUM_GPUS >= 1 else "cpu"
GPU_BATCH_FRACTION = 0.92
VAL_BATCH = max(16, 32 * max(1, NUM_GPUS)) if NUM_GPUS else 8
PREDICT_BATCH = max(8, 32 * max(1, NUM_GPUS)) if NUM_GPUS else 8
WORKERS = min(8, os.cpu_count() or 2)

RUNS_DIR = REPO_DIR / "runs"
RUN_NAME = "yolov8s_t4"
BASE_WEIGHTS = MODELS_DIR / "yolov8s.pt"
BEST_WEIGHTS = MODELS_DIR / "best.pt"
LAST_WEIGHTS = MODELS_DIR / "last.pt"
VAL_PREDICTIONS_JSON = REPO_DIR / "val_predictions.json"
VAL_METRICS_JSON = REPO_DIR / "val_metrics.json"

TRAIN_ARGS = {
    "data": str(DATA_YAML),
    "epochs": 80,
    "imgsz": 640,
    "batch": GPU_BATCH_FRACTION if NUM_GPUS else 16,
    "patience": 20,
    "workers": WORKERS,
    "device": TRAIN_DEVICE,
    "amp": bool(NUM_GPUS),
    "cache": "disk",
    "optimizer": "auto",
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": True,
    "seed": 42,
    "deterministic": True,
    "close_mosaic": 10,
    "plots": True,
    "verbose": True,
}

print(f"TRAIN_DEVICE: {TRAIN_DEVICE}")
print(f"INFER_DEVICE: {INFER_DEVICE}")
print(f"Auto-batch target fraction: {GPU_BATCH_FRACTION if NUM_GPUS else 'CPU fixed batch'}")
print(f"Validation batch: {VAL_BATCH}")
print(f"Prediction batch: {PREDICT_BATCH}")
print(f"Workers per rank: {WORKERS}")

In [ ]:
VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def ensure_base_weights(target_path: Path) -> Path:
    if target_path.exists():
        return target_path

    _ = YOLO("yolov8s.pt")
    cache_dir = Path.home() / ".cache" / "ultralytics"
    candidates = [REPO_DIR / "yolov8s.pt", Path("yolov8s.pt")]

    if cache_dir.exists():
        candidates.extend(cache_dir.rglob("yolov8s.pt"))

    for candidate in candidates:
        if candidate.exists():
            shutil.copy2(candidate, target_path)
            return target_path

    raise FileNotFoundError("Could not locate yolov8s.pt after download.")


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def reset_label_dir(label_dir: Path) -> None:
    label_dir.mkdir(parents=True, exist_ok=True)
    for txt_path in label_dir.glob("*.txt"):
        txt_path.unlink()


def reset_error_dir(error_dir: Path) -> None:
    error_dir.mkdir(parents=True, exist_ok=True)
    for path in error_dir.iterdir():
        if path.name == ERROR_JSON.name:
            continue
        if path.is_file():
            path.unlink()
        elif path.is_dir():
            shutil.rmtree(path)


# The dataset uses 1-based inclusive xyxy boxes, e.g. [1, 1, width, height].
# Convert them to normalized YOLO cx cy w h labels.
def xyxy_1based_to_yolo(box, width: int, height: int):
    x1, y1, x2, y2 = [float(value) for value in box]

    x1 = min(max(x1, 1.0), float(width))
    y1 = min(max(y1, 1.0), float(height))
    x2 = min(max(x2, x1), float(width))
    y2 = min(max(y2, y1), float(height))

    x1_zero = x1 - 1.0
    y1_zero = y1 - 1.0
    box_w = max(x2 - x1_zero, 1.0)
    box_h = max(y2 - y1_zero, 1.0)
    center_x = x1_zero + box_w / 2.0
    center_y = y1_zero + box_h / 2.0

    return (
        center_x / width,
        center_y / height,
        box_w / width,
        box_h / height,
    )


def resolve_image_path(image_info: dict) -> Path:
    image_path = PUBLIC_DIR / image_info["file_name"]
    if not image_path.exists():
        raise FileNotFoundError(f"Missing image file: {image_path}")
    if image_path.suffix.lower() not in VALID_IMAGE_EXTS:
        raise ValueError(f"Unsupported image extension: {image_path.suffix}")
    return image_path


def build_image_records(split_data: dict):
    records = []
    for image_info in split_data["images"]:
        image_path = resolve_image_path(image_info)
        records.append(
            {
                "image_id": image_info["id"],
                "path": image_path,
                "file_name": image_info["file_name"],
                "width": int(image_info["width"]),
                "height": int(image_info["height"]),
            }
        )
    return records


def write_yolo_labels(annotation_path: Path, label_dir: Path, class_to_idx: dict[str, int]):
    data = load_json(annotation_path)
    reset_label_dir(label_dir)

    image_by_id = {item["id"]: item for item in data["images"]}
    annotations_by_image = defaultdict(list)
    for ann in data["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    for image_id, image_info in image_by_id.items():
        width = int(image_info["width"])
        height = int(image_info["height"])
        lines = []

        for ann in annotations_by_image.get(image_id, []):
            class_id = class_to_idx[ann["class"]]
            x, y, w, h = xyxy_1based_to_yolo(ann["bbox"], width, height)
            lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

        label_path = label_dir / f"{Path(image_id).stem}.txt"
        label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

    return data


classes = load_json(CLASSES_JSON)
class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}

train_data = write_yolo_labels(TRAIN_JSON, TRAIN_LABEL_DIR, class_to_idx)
val_data = write_yolo_labels(VAL_JSON, VAL_LABEL_DIR, class_to_idx)
train_image_records = build_image_records(train_data)
val_image_records = build_image_records(val_data)

DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(PUBLIC_DIR),
            "train": "train/images",
            "val": "val/images",
            "names": {idx: name for idx, name in enumerate(classes)},
        },
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)

BASE_WEIGHTS = ensure_base_weights(BASE_WEIGHTS)
ERROR_JSON.write_text("[]\n", encoding="utf-8")

print(DATA_YAML.read_text(encoding="utf-8"))
print(f"Train images: {len(train_data['images'])}, train boxes: {len(train_data['annotations'])}")
print(f"Val images: {len(val_data['images'])}, val boxes: {len(val_data['annotations'])}")
print(f"Base weights: {BASE_WEIGHTS}")
print(f"Classes: {classes}")
print(f"Error JSON initialized at: {ERROR_JSON}")

In [ ]:
def show_annotation_samples(split_data: dict, max_images: int = 4, seed: int = 42):
    image_by_id = {item["id"]: item for item in split_data["images"]}
    annotations_by_image = defaultdict(list)
    for ann in split_data["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    populated_image_ids = [image_id for image_id, anns in annotations_by_image.items() if anns]
    random.seed(seed)
    sample_ids = random.sample(populated_image_ids, k=min(max_images, len(populated_image_ids)))

    fig, axes = plt.subplots(1, len(sample_ids), figsize=(5 * len(sample_ids), 5))
    if len(sample_ids) == 1:
        axes = [axes]

    for axis, image_id in zip(axes, sample_ids):
        image_info = image_by_id[image_id]
        image_path = resolve_image_path(image_info)
        image = Image.open(image_path).convert("RGB")
        drawer = ImageDraw.Draw(image)

        for ann in annotations_by_image[image_id]:
            x1, y1, x2, y2 = ann["bbox"]
            drawer.rectangle([x1 - 1, y1 - 1, x2 - 1, y2 - 1], outline="red", width=3)
            drawer.text((x1, max(1, y1 - 18)), ann["class"], fill="yellow")

        axis.imshow(image)
        axis.set_title(image_id)
        axis.axis("off")

    plt.tight_layout()


show_annotation_samples(train_data)
print("Train class distribution:")
print(Counter(ann["class"] for ann in train_data["annotations"]))

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = YOLO(str(BASE_WEIGHTS))
train_results = model.train(**TRAIN_ARGS)

run_dir = RUNS_DIR / RUN_NAME
run_weights_dir = run_dir / "weights"
run_best = run_weights_dir / "best.pt"
run_last = run_weights_dir / "last.pt"

if run_best.exists():
    shutil.copy2(run_best, BEST_WEIGHTS)
if run_last.exists():
    shutil.copy2(run_last, LAST_WEIGHTS)

print(f"Run directory: {run_dir}")
print(f"Best checkpoint copied to: {BEST_WEIGHTS}")
print(f"Last checkpoint copied to: {LAST_WEIGHTS}")

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML), split="val", device=INFER_DEVICE, batch=VAL_BATCH)

summary = {
    "mAP50-95": float(metrics.box.map),
    "mAP50": float(metrics.box.map50),
    "mAP75": float(metrics.box.map75),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
}
summary

In [ ]:
def yolo_xyxy_to_project_box(box, width: int, height: int):
    x1, y1, x2, y2 = [float(value) for value in box]

    xmin = min(max(x1 + 1.0, 1.0), float(width))
    ymin = min(max(y1 + 1.0, 1.0), float(height))
    xmax = min(max(x2, xmin), float(width))
    ymax = min(max(y2, ymin), float(height))

    return [round(xmin, 4), round(ymin, 4), round(xmax, 4), round(ymax, 4)]


val_image_paths = [str(record["path"]) for record in val_image_records]
project_predictions = []

for result in best_model.predict(
    source=val_image_paths,
    imgsz=TRAIN_ARGS["imgsz"],
    conf=0.25,
    iou=0.7,
    batch=PREDICT_BATCH,
    half=bool(NUM_GPUS),
    device=INFER_DEVICE,
    verbose=False,
    stream=True,
):
    image_id = Path(result.path).name
    image_height, image_width = result.orig_shape
    boxes = []

    if result.boxes is not None and len(result.boxes) > 0:
        xyxy_list = result.boxes.xyxy.cpu().tolist()
        conf_list = result.boxes.conf.cpu().tolist()
        cls_list = result.boxes.cls.cpu().tolist()

        for box, confidence, class_id in zip(xyxy_list, conf_list, cls_list):
            boxes.append(
                {
                    "class": classes[int(class_id)],
                    "confidence": round(float(confidence), 6),
                    "bbox": yolo_xyxy_to_project_box(box, image_width, image_height),
                }
            )

    project_predictions.append({"image_id": image_id, "boxes": boxes})

VAL_PREDICTIONS_JSON.write_text(
    json.dumps(project_predictions, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

subprocess.run(
    [
        sys.executable,
        str(PUBLIC_DIR / "tools" / "evaluate_predictions.py"),
        "--ground_truth",
        str(VAL_JSON),
        "--predictions",
        str(VAL_PREDICTIONS_JSON),
        "--output",
        str(VAL_METRICS_JSON),
    ],
    check=True,
)

print(f"Saved predictions: {VAL_PREDICTIONS_JSON}")
print(f"Saved metrics: {VAL_METRICS_JSON}")
print(VAL_METRICS_JSON.read_text(encoding="utf-8"))

In [ ]:
def bbox_iou(box_a, box_b) -> float:
    ax1, ay1, ax2, ay2 = [float(v) for v in box_a]
    bx1, by1, bx2, by2 = [float(v) for v in box_b]

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def draw_project_box(drawer: ImageDraw.ImageDraw, bbox, color: str, text: str, width: int = 3):
    x1, y1, x2, y2 = [float(v) for v in bbox]
    drawer.rectangle([x1 - 1, y1 - 1, x2 - 1, y2 - 1], outline=color, width=width)
    drawer.text((x1, max(1, y1 - 18)), text, fill=color)


def export_error_cases(split_data: dict, predictions: list[dict], iou_threshold: float = 0.5):
    reset_error_dir(ERROR_DIR)

    image_by_id = {item["id"]: item for item in split_data["images"]}
    gt_by_image = defaultdict(list)
    for ann in split_data["annotations"]:
        gt_by_image[ann["image_id"]].append({"class": ann["class"], "bbox": ann["bbox"]})

    pred_by_image = {item["image_id"]: item["boxes"] for item in predictions}
    error_cases = []

    for image_id, image_info in image_by_id.items():
        gt_objects = gt_by_image.get(image_id, [])
        pred_objects = sorted(pred_by_image.get(image_id, []), key=lambda item: item.get("confidence", 0.0), reverse=True)
        matched_pred_indexes = set()
        missing_objects = []

        for gt in gt_objects:
            best_idx = -1
            best_iou = 0.0
            for pred_idx, pred in enumerate(pred_objects):
                if pred_idx in matched_pred_indexes or pred["class"] != gt["class"]:
                    continue
                iou = bbox_iou(gt["bbox"], pred["bbox"])
                if iou > best_iou:
                    best_iou = iou
                    best_idx = pred_idx

            if best_idx >= 0 and best_iou >= iou_threshold:
                matched_pred_indexes.add(best_idx)
            else:
                missing_objects.append(gt)

        if not missing_objects:
            continue

        image_path = resolve_image_path(image_info)
        rendered = Image.open(image_path).convert("RGB")
        drawer = ImageDraw.Draw(rendered)

        for gt in gt_objects:
            draw_project_box(drawer, gt["bbox"], color="lime", text=f"GT:{gt['class']}", width=2)
        for pred in pred_objects:
            draw_project_box(drawer, pred["bbox"], color="cyan", text=f"PR:{pred['class']} {pred['confidence']:.2f}", width=2)
        for miss in missing_objects:
            draw_project_box(drawer, miss["bbox"], color="red", text=f"MISS:{miss['class']}", width=4)

        error_image_path = ERROR_DIR / image_id
        rendered.save(error_image_path)

        error_cases.append(
            {
                "image_id": image_id,
                "source_image": image_info["file_name"],
                "saved_error_image": str(error_image_path.relative_to(REPO_DIR)).replace('\\', '/'),
                "ground_truth_count": len(gt_objects),
                "predicted_count": len(pred_objects),
                "matched_count": len(gt_objects) - len(missing_objects),
                "missing_count": len(missing_objects),
                "missing_objects": missing_objects,
            }
        )

    ERROR_JSON.write_text(json.dumps(error_cases, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return error_cases


error_cases = export_error_cases(val_data, project_predictions, iou_threshold=0.5)
print(f"Error images exported: {len(error_cases)}")
print(f"Error JSON: {ERROR_JSON}")
error_cases[:3]

In [ ]:
preview_paths = [Path(record["path"]) for record in val_image_records[:4]]
preview_title = "Validation sample images"
if ERROR_JSON.exists():
    saved_error_cases = json.loads(ERROR_JSON.read_text(encoding="utf-8"))
    if saved_error_cases:
        preview_paths = [REPO_DIR / item["saved_error_image"] for item in saved_error_cases[:4]]
        preview_title = "Exported error images"

fig, axes = plt.subplots(1, len(preview_paths), figsize=(6 * len(preview_paths), 6))
if len(preview_paths) == 1:
    axes = [axes]

for axis, image_path in zip(axes, preview_paths):
    axis.imshow(Image.open(image_path).convert("RGB"))
    axis.set_title(image_path.name)
    axis.axis("off")

plt.suptitle(preview_title)
plt.tight_layout()

print("Artifacts:")
print(f"- Base weights: {BASE_WEIGHTS}")
print(f"- Best weights: {BEST_WEIGHTS}")
print(f"- Last weights: {LAST_WEIGHTS}")
print(f"- YOLO data config: {DATA_YAML}")
print(f"- Val predictions: {VAL_PREDICTIONS_JSON}")
print(f"- Val metrics: {VAL_METRICS_JSON}")
print(f"- Error JSON: {ERROR_JSON}")
print(f"- Error image dir: {ERROR_DIR}")